# PA1 — Inferência de instâncias

Recebe o caminho de **qualquer imagem** (PNG/JPG/TIF, cinza ou RGB, qualquer tamanho), devolve a máscara de instâncias colorida e a contagem. **Não retreina**: usa o checkpoint do modelo final em `runs/dsb_p2_unet_boundary/model.pt` (U-Net treinada no DSB2018, cabeça fronteira + distância, decodificação por watershed). Imagens maiores que 256 px são processadas em tiles com costura de mapas (correção B da Parte 4).

Rode as células em ordem. Só mude `IMAGE_PATH`. O exemplo `exemplo_dsb.png` é uma imagem do **teste** do DSB2018 (fluorescência, 520×696, 89 núcleos); `exemplo.png` é do dataset sintético da Parte 0 (use `RUN = 'runs/p2_unet_boundary'` para ele).

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '.')                      # raiz do repositório
from pa1_instseg.infer import infer_image
from pa1_instseg.viz import color_instances, to_display

RUN = 'runs/dsb_p2_unet_boundary'           # checkpoint do modelo final (DSB2018)
IMAGE_PATH = 'exemplo_dsb.png'               # <- troque pelo caminho da sua imagem
MARKER_SOURCE = 'dist'                        # 'interior' (padrão da Parte 2) ou 'dist' (correção da Parte 5)

In [ ]:
r = infer_image(RUN, IMAGE_PATH, decode_kw={'marker_source': MARKER_SOURCE})
print(f"{r['count']} instâncias encontradas")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(to_display(r['image']).squeeze(), cmap='gray', vmin=0, vmax=1); ax[0].set_title('imagem')
ax[1].imshow(r['maps'][2], cmap='magma', vmin=0, vmax=1);          ax[1].set_title('p(fronteira)')
ax[2].imshow(r['maps'][3], cmap='viridis', vmin=0, vmax=1);        ax[2].set_title('distância prevista')
ax[3].imshow(r['colored']);                                          ax[3].set_title(f"instâncias ({r['count']})")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Salvar a máscara

`r['instances']` é um `int32` H×W com ids 1..N (0 = fundo). A versão colorida vai para PNG:

In [ ]:
from PIL import Image
out = IMAGE_PATH.rsplit('.', 1)[0] + '_instancias.png'
Image.fromarray(r['colored']).save(out)
np.save(IMAGE_PATH.rsplit('.', 1)[0] + '_instancias.npy', r['instances'])
print('salvo em', out)

## (Opcional) comparar com o ground truth

Se você tiver o mapa de instâncias verdadeiro (`.npy` H×W com ids), as métricas da Parte 1 — implementadas do zero em `pa1_instseg/metrics.py` — são:

In [ ]:
from pa1_instseg.metrics import instance_metrics, THRESHOLDS
import os
gt_path = IMAGE_PATH.rsplit('.', 1)[0] + '_gt.npy'
if os.path.exists(gt_path):
    gt = np.load(gt_path)
    m = instance_metrics(r['instances'], gt, rule='greedy')
    print(f"mAP@[.5:.95] = {m['map']:.3f}   AP50 = {m['ap50']:.3f}   GT = {m['n_gt']}  pred = {m['n_pred']}  |erro| = {m['count_err']}")
    print('AP por limiar:', dict(zip(THRESHOLDS.tolist(), m['ap'].round(2).tolist())))
else:
    print('sem ground truth para', IMAGE_PATH)